In [0]:
# Step 1: Set up the Silver source and Gold tables
from pyspark.sql import functions as F

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

silver_table = f"{catalog}.{schema_name}.streaming_orders_silver"

daily_table = f"{catalog}.{schema_name}.streaming_gold_daily_sales"
product_table = f"{catalog}.{schema_name}.streaming_gold_product_performance"
regional_table = f"{catalog}.{schema_name}.streaming_gold_regional_sales"

silver_df = spark.table(silver_table)

print("Silver schema:")
silver_df.printSchema()
display(silver_df.limit(100))

In [0]:
# Step 2: Build daily sales metrics
completed_df = silver_df.filter(
    F.lower(F.trim(F.col("status"))) == "completed"
)

daily_sales_df = (
    completed_df
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.sum(F.col("quantity") * F.col("unit_price")), 2
        ).alias("total_revenue")
    )
    .withColumn("average_order_value",
                F.round(F.col("total_revenue") / F.col("total_orders"), 2))
)

display(daily_sales_df.orderBy("order_date"))

In [0]:
# Step 3: Compare products by revenue and volume
product_sales_df = (
    completed_df
    .groupBy("product_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.sum(F.col("quantity") * F.col("unit_price")), 2
        ).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

display(product_sales_df)

In [0]:
# Step 4: Compare sales across regions
regional_sales_df = (
    completed_df
    .groupBy("region")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.sum(F.col("quantity") * F.col("unit_price")), 2
        ).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

display(regional_sales_df)

In [0]:
# Step 5: Save the analytics tables in Delta format
daily_sales_df.write.format("delta").mode("overwrite").saveAsTable(daily_table)

product_sales_df.write.format("delta").mode("overwrite").saveAsTable(product_table)

regional_sales_df.write.format("delta").mode("overwrite").saveAsTable(regional_table)

print("Gold analytics tables created successfully.")

In [0]:
# Step 6: Calculate overall retail KPIs
kpi_df = completed_df.agg(
    F.countDistinct("order_id").alias("total_completed_orders"),
    F.sum("quantity").alias("total_units_sold"),
    F.round(
        F.sum(F.col("quantity") * F.col("unit_price")), 2
    ).alias("total_revenue")
).withColumn(
    "average_order_value",
    F.round(
        F.col("total_revenue") / F.col("total_completed_orders"), 2
    )
)

display(kpi_df)

In [0]:
# Step 7: Confirm that the Gold tables were saved
for table_name in [daily_table, product_table, regional_table]:
    print(f"{table_name}: {spark.table(table_name).count()} rows")